### Comparing to Mattergen

Evaluates Mattergen's conditional generation head-to-head with CrystaLLM-Prefix trained from scratch. Both models are trained and evaluated on `c-bone/mpdb-2prop_clean`, using identical generation targets and the same VSUN / property metrics pipeline.

> **Some Key un-changeable differences vs. CrystaLLM-pi**
> 1. Mattergen trains on Reduced Niggli structures; we train on Conventional CIFs.
> 2. Conditional generation in Mattergen requires a two-phase pipeline: unconditional base then adapter fine-tune whereas CrystaLLM-pi learns the conditional distribution in a single pass.
> 3. Closest available Mattergen config is `mp-20` (batch 512, ReduceLROnPlateau, 900 epochs), ours uses batch 32 with CosineDecayWithMinLR for 24 epochs.
> 4. Needless to say the architectures are nothing alike :)

### 1 · Dataset Preparation

Convert `c-bone/mpdb-2prop_clean` to the CSV format expected by Mattergen, preserving train/validation splits.

In [ ]:
import __init__
import datasets
from _utils import process_dataframe, filter_df_to_context

# load the dataset c-bone/mpdb-2prop_clean
dataset = datasets.load_dataset("c-bone/mpdb-2prop_clean")

# load as a dataframe
df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

dfs = {
    "df_train": df_train,
    "df_val": df_val
}

for name, df in dfs.items():
    print(f"{name}: {df.shape}")

    print(f"length before filtering: {len(df)}")
    df = filter_df_to_context(df=df, context=1024, num_workers=64)
    print(f"after filtering: {len(df)}")

    df_clean = process_dataframe(df=df, num_workers=64, column_name="CIF")

    df_clean = df_clean.rename(columns={
        'CIF': 'cif',
        'Database': 'material_id',
        'Bandgap (eV)': 'dft_band_gap',
        'Energy Above Hull (eV)': 'energy_above_hull'
    })
    df_clean['material_id'] = [f"id_{i}" for i in range(len(df_clean))]
    df_clean = df_clean[["material_id", "cif", "dft_band_gap", "energy_above_hull"]]
    df_clean.to_csv(f'/home/cyprien/mattergen/datasets/mp_bandgap/{name}.csv', index=False)


### 2 · Atom Count Distribution

Mattergen samples unit-cell size from the training distribution during generation. Compute the distribution below and paste the output into the generation config.

In [ ]:
import __init__
from _utils._notebook_utils.b1b_mattergen_utils import compute_atom_counts, load_count_datasets, compute_atom_count_distribution

loaded = load_count_datasets([("MP Bandgap", "HF-databases/mpdb_processing/mpdb_2prop_count.parquet", 1024)])
loaded = compute_atom_counts(loaded)
atom_count_distributions = compute_atom_count_distribution(loaded[0][1])

print(atom_count_distributions)


### 3 · Training

Two-phase training using the `mp-20` Mattergen config adapted for this dataset.

- **Phase 1**:  unconditional base: 900 epochs
- **Phase 2**: band gap CFG adapter fine-tune: 200 epochs

Exact configs used: `_config_files/training/conditional/pretraining_benefits/mattergen_configs/`


### 4 · Generation

Generate structures conditioned on the same band-gap targets used in Notebook B1a (same number of samples per target). Atom-count distribution from Cell 2 is set in the generation config.

Config path: `_config_files/generation/conditional/pretraining_benefits/mattergen_configs/`


### 5 · Post-processing & Metrics

Steps mirror the CrystaLLM pipeline exactly. Mattergen outputs must be converted to a `Generated CIF` column before this section runs (one row per generated structure, same format as CrystaLLM output).

In [ ]:
!python _utils/_metrics/VUN_metrics.py \
    --input_parquet '/home/cyprien/mattergen/gen_tests/proc/mattergen-scratch-cond_gen.parquet' \
    --huggingface_dataset 'c-bone/mpdb-2prop_clean' \
    --load_processed_data 'HF-databases/mpdb-2prop_clean/mpdb_2prop_proc.parquet' \
    --output_parquet '/home/cyprien/mattergen/gen_tests/proc/mattergen-scratch-cond_metrics.parquet' \
    --num_workers 32

In [ ]:
import __init__
import pandas as pd
from _utils._notebook_utils.b1b_mattergen_utils import (
    compute_atom_counts,
    annotate_structural_features,
    annotate_spacegroups_by_symprec,
)

df_mg = pd.read_parquet('/home/cyprien/mattergen/gen_tests/proc/mattergen-scratch-cond_metrics.parquet')
loaded_mg = [("Mattergen", df_mg, 1024)]
loaded_mg = compute_atom_counts(loaded_mg)
df_mg = loaded_mg[0][1]

df_mg = annotate_structural_features(df_mg)
df_mg = annotate_spacegroups_by_symprec(df_mg, symprecs=(0.01, 0.1, 0.2), num_workers=32)
df_mg.to_parquet('/home/cyprien/mattergen/gen_tests/proc/mattergen-scratch-cond_xtra-metrics.parquet', index=False)


In [ ]:
!python _utils/_metrics/mace_ehull.py \
    --post_parquet '/home/cyprien/mattergen/gen_tests/proc/mattergen-scratch-cond_xtra-metrics.parquet' \
    --output_parquet '/home/cyprien/mattergen/gen_tests/proc/mattergen-scratch-cond_xtra-metrics.parquet' \
    --num_workers 16 \
    --batch_size 10 \
    --apply_system_type_filter 6

> Note: When running this, I removed the rows where system size > 6, this is because of the combinatorial explosion of phase diagram construction which was impossible to compute with our available hardware.
> 
> Generally, most physical structures would not go beyong senary, apart for some high-entropy alloys, but even then it would be rare to see anything useful in those ranges

In [ ]:
!python _utils/_metrics/property_metrics.py \
    --post_parquet '/home/cyprien/mattergen/gen_tests/proc/mattergen-scratch-cond_xtra-metrics.parquet' \
    --output_parquet '/home/cyprien/mattergen/gen_tests/proc/mattergen-scratch-cond_xtra-metrics.parquet' \
    --property_targets '["Bandgap (eV)", "Energy Above Hull (eV)"]' \
    --num_workers 16 \
    --property1_normaliser "power_log" \
    --property2_normaliser "linear" \
    --max_property1 17.891 \
    --min_property1 0.0 \
    --max_property2 5.418 \
    --min_property2 0.0

#### CrystaLLM Reference Annotation

Annotate the CrystaLLM-Prefix scratch parquet with atom counts and structural features for the distribution comparison plots. Results are cached; subsequent runs are instant.

In [ ]:
import os

ANNOTATED_PKV = '_artifacts/pretrain_benefits/scratch-methods/mpdb-scratch-PKV_xtra-post.parquet'
REQUIRED_SPG_COLS = ['spacegroup_symprec_0p01', 'spacegroup_symprec_0p1', 'spacegroup_symprec_0p2']

if os.path.exists(ANNOTATED_PKV):
    df_pkv_ref = pd.read_parquet(ANNOTATED_PKV)
    print(f"Loaded cached PKV annotations ({len(df_pkv_ref):,} rows).")
else:
    df_pkv_ref = pd.read_parquet('_artifacts/pretrain_benefits/scratch-methods/mpdb-scratch-PKV_post.parquet')
    loaded = [("PKV", df_pkv_ref, 1024)]
    loaded = compute_atom_counts(loaded)
    df_pkv_ref = annotate_structural_features(loaded[0][1])

missing = [c for c in REQUIRED_SPG_COLS if c not in df_pkv_ref.columns]
if missing:
    df_pkv_ref = annotate_spacegroups_by_symprec(df_pkv_ref, symprecs=(0.01, 0.1, 0.2), num_workers=32)
    df_pkv_ref.to_parquet(ANNOTATED_PKV, index=False)
    print(f"Updated PKV parquet with symmetry sweep columns: {missing}")
else:
    print("PKV symmetry sweep columns already present.")

### 6 · Results & Plots

In [ ]:
import __init__
import os
import pandas as pd
from datasets import load_dataset
from _utils._notebook_utils.b1b_mattergen_utils import (
    plot_mg_vs_pkv_parity_grid,
    format_mg_vs_pkv_metrics_table,
    plot_mg_vs_pkv_output_space_summary,
    plot_mg_structural_distributions,
)


os.makedirs('plots/mattergen', exist_ok=True)

df_pkv = pd.read_parquet('_artifacts/pretrain_benefits/scratch-methods/mpdb-scratch-PKV_xtra-post.parquet')
df_mg  = pd.read_parquet('/home/cyprien/mattergen/gen_tests/proc/mattergen-scratch-cond_xtra-metrics.parquet')

ds = load_dataset('c-bone/mpdb-2prop_clean', split='train')
train_df = ds.to_pandas()

In [ ]:
fig, metrics_df = plot_mg_vs_pkv_parity_grid(df_pkv, df_mg, train_df=train_df)

# save fig
fig.savefig('plots/mattergen/parity_grid.png', dpi=300, bbox_inches='tight')

formatted_table = format_mg_vs_pkv_metrics_table(metrics_df)
formatted_table.to_csv(
    'plots/mattergen/parity_grid_metrics-025.csv',
    index=False,
 )
with open('plots/mattergen/parity_grid_metrics-025.tex', 'w') as handle:
    handle.write(formatted_table.to_latex(index=False, escape=False))
print("\nVSUN Parity Grid Metrics Table:")
print(formatted_table.to_string(index=False))

In [ ]:
fig2 = plot_mg_vs_pkv_output_space_summary(df_pkv, df_mg, train_df=train_df)
fig2.savefig('plots/mattergen/output_space_summary.png', dpi=300, bbox_inches='tight')

In [ ]:
import __init__
import os
import pandas as pd
from _utils._notebook_utils.b1b_mattergen_utils import (
    compute_atom_counts,
    annotate_structural_features,
    annotate_spacegroups_by_symprec,
)

TRAIN_STRUCT_REF = '_artifacts/pretrain_benefits/mpdb-2prop_train_structural_ref.parquet'
REQUIRED_TRAIN_COLS = [
    'conv_count',
    'system_type',
    'spacegroup_symprec_0p1',
]

if os.path.exists(TRAIN_STRUCT_REF):
    train_struct_df = pd.read_parquet(TRAIN_STRUCT_REF)
    print(f"Loaded cached train structural reference ({len(train_struct_df):,} rows).")
else:
    train_struct_df = pd.read_parquet('HF-databases/mpdb-2prop_clean/mpdb_2prop_proc.parquet')

if 'conv_count' not in train_struct_df.columns or 'system_type' not in train_struct_df.columns:
    loaded_train = [("Train set", train_struct_df, 1024)]
    loaded_train = compute_atom_counts(loaded_train)
    train_struct_df = annotate_structural_features(loaded_train[0][1], cif_col='CIF', num_workers=32)

missing_train_sg = [c for c in REQUIRED_TRAIN_COLS if c not in train_struct_df.columns]
if missing_train_sg:
    train_struct_df = annotate_spacegroups_by_symprec(train_struct_df, cif_col='CIF', symprecs=([0.1]), num_workers=32)

if missing_train_sg or (not os.path.exists(TRAIN_STRUCT_REF)):
    train_struct_df.to_parquet(TRAIN_STRUCT_REF, index=False)
    print("Saved updated train structural reference cache.")
else:
    print("Train structural reference already up to date.")

In [ ]:
fig3 = plot_mg_structural_distributions(df_pkv, df_mg, train_df=train_struct_df)
fig3.savefig('plots/mattergen/structural_distributions.png', dpi=300, bbox_inches='tight')